# Age-Standardized Violence Target Analysis

This notebook constructs and analyzes an age-standardized violence target using the uploaded `violence_target` extract.

Target definition requested:

- Age window: 15-25.
- Eligibility: observed through at least age 25 in the violence-item panel.
- Positive: at least 2 distinct age-years with a qualifying violence event between ages 15 and 25.
- Predictor cutoff: age 15 for every person. Predictors should use only information known before the 15th birthday.
- Negative: people observed through age 25 with 0 or 1 qualifying event age-year are valid zeros.

Important note: ages are approximate because the extract contains birth year and survey year, not exact birth date. Age is computed as `survey_year - birth_year`.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("Matplotlib not available:", exc)

try:
    from IPython.display import display as safe_display
except Exception:
    def safe_display(x):
        print(x)

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "violence_target":
    PROJECT_DIR = PROJECT_DIR.parent

VIOLENCE_DIR = PROJECT_DIR / "violence_target"
OUT_DIR = VIOLENCE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

VIOLENCE_PATH = VIOLENCE_DIR / "violence_target.csv"
MAIN_FEATURES_PATH = PROJECT_DIR / "nlsy79_child_youngadult_selected_crime_features.csv"

AGE_START = 15
AGE_END = 25
PREDICTOR_CUTOFF_AGE = 15
MISSING_CODES = {-1, -2, -3, -4, -5, -7}

print("Project dir:", PROJECT_DIR)
print("Violence extract:", VIOLENCE_PATH)
print("Main feature file:", MAIN_FEATURES_PATH)


Project dir: /Users/diergexiaohai/Desktop/Bol_Crime
Violence extract: /Users/diergexiaohai/Desktop/Bol_Crime/violence_target/violence_target.csv
Main feature file: /Users/diergexiaohai/Desktop/Bol_Crime/nlsy79_child_youngadult_selected_crime_features.csv


## Violence Item Metadata

The extract contains violence/aggression items across survey years. A qualifying event is coded when any item in a survey year indicates physical fighting, attacking someone, or hurting someone badly enough to require medical attention.

For yes/no variables, `1` is positive. For count/frequency variables, `0 = never` or `1 = never` depending on the item family; any non-missing value above the never category is positive.


In [2]:
ITEMS = [
    # Child self-administered items
    {"csv_code": "C0730700", "ref_id": "C07307.00", "variable": "CS884221", "survey_year": 1988, "item_family": "hurt_need_doctor", "positive_rule": "value >= 1", "question": "Hurt someone bad enough to need a doctor"},
    {"csv_code": "C0942400", "ref_id": "C09424.00", "variable": "CS906613", "survey_year": 1990, "item_family": "hurt_need_doctor", "positive_rule": "value >= 1", "question": "Number of times hurt someone enough to need a doctor"},

    # Young adult yes/no items: 1 yes, 0 no
    {"csv_code": "Y0375300", "ref_id": "Y03753.00", "variable": "YASR-61B", "survey_year": 1994, "item_family": "physical_fight", "positive_rule": "value == 1", "question": "In past year got in fight at school or work"},
    {"csv_code": "Y0375900", "ref_id": "Y03759.00", "variable": "YASR-61I", "survey_year": 1994, "item_family": "attacked_to_hurt", "positive_rule": "value == 1", "question": "In past year attacked someone to seriously hurt them"},
    {"csv_code": "Y0376500", "ref_id": "Y03765.00", "variable": "YASR-61O", "survey_year": 1994, "item_family": "hurt_need_doctor", "positive_rule": "value == 1", "question": "In past year hurt someone enough to need doctor"},
    {"csv_code": "Y0669000", "ref_id": "Y06690.00", "variable": "YASR-61B", "survey_year": 1996, "item_family": "physical_fight", "positive_rule": "value == 1", "question": "In past year got in fight at school or work"},
    {"csv_code": "Y0669600", "ref_id": "Y06696.00", "variable": "YASR-61I", "survey_year": 1996, "item_family": "attacked_to_hurt", "positive_rule": "value == 1", "question": "In past year attacked someone to seriously hurt them"},
    {"csv_code": "Y0670200", "ref_id": "Y06702.00", "variable": "YASR-61O", "survey_year": 1996, "item_family": "hurt_need_doctor", "positive_rule": "value == 1", "question": "In past year hurt someone enough to need doctor"},
    {"csv_code": "Y0966400", "ref_id": "Y09664.00", "variable": "YASR-61B", "survey_year": 1998, "item_family": "physical_fight", "positive_rule": "value == 1", "question": "In past year got in fight at school or work"},
    {"csv_code": "Y0967000", "ref_id": "Y09670.00", "variable": "YASR-61I", "survey_year": 1998, "item_family": "attacked_to_hurt", "positive_rule": "value == 1", "question": "In past year attacked someone to seriously hurt them"},
    {"csv_code": "Y0967600", "ref_id": "Y09676.00", "variable": "YASR-61O", "survey_year": 1998, "item_family": "hurt_need_doctor", "positive_rule": "value == 1", "question": "In past year hurt someone enough to need doctor"},
    {"csv_code": "Y1176200", "ref_id": "Y11762.00", "variable": "YASR-61B", "survey_year": 2000, "item_family": "physical_fight", "positive_rule": "value == 1", "question": "In last year got in physical fight at school or work"},

    # Young adult count/frequency items: 1 never, 2+ positive
    {"csv_code": "Y1415900", "ref_id": "Y14159.00", "variable": "YASR-60C", "survey_year": 2002, "item_family": "hurt_physically", "positive_rule": "value >= 2", "question": "Times in last year hurt someone physically"},
    {"csv_code": "Y1416800", "ref_id": "Y14168.00", "variable": "YASR-61B", "survey_year": 2002, "item_family": "physical_fight", "positive_rule": "value == 1", "question": "In last year got in physical fight at school or work"},
    {"csv_code": "Y1667300", "ref_id": "Y16673.00", "variable": "YASR-60C", "survey_year": 2004, "item_family": "hurt_physically", "positive_rule": "value >= 2", "question": "Times in last year hurt someone physically"},
    {"csv_code": "Y1668200", "ref_id": "Y16682.00", "variable": "YASR-61B", "survey_year": 2004, "item_family": "physical_fight", "positive_rule": "value == 1", "question": "In last year got in physical fight at school or work"},
    {"csv_code": "Y1940600", "ref_id": "Y19406.00", "variable": "YASR-60C", "survey_year": 2006, "item_family": "hurt_physically", "positive_rule": "value >= 2", "question": "Times in last year hurt someone physically"},
    {"csv_code": "Y1941500", "ref_id": "Y19415.00", "variable": "YASR-61B", "survey_year": 2006, "item_family": "physical_fight", "positive_rule": "value == 1", "question": "In last year got in physical fight at school or work"},
    {"csv_code": "Y2256600", "ref_id": "Y22566.00", "variable": "YASR-60C", "survey_year": 2008, "item_family": "hurt_physically", "positive_rule": "value >= 2", "question": "Times in last year hurt someone physically"},
    {"csv_code": "Y2257500", "ref_id": "Y22575.00", "variable": "YASR-61B", "survey_year": 2008, "item_family": "physical_fight", "positive_rule": "value == 1", "question": "In last year got in physical fight at school or work"},
    {"csv_code": "Y2608200", "ref_id": "Y26082.00", "variable": "YASR-60C", "survey_year": 2010, "item_family": "hurt_physically", "positive_rule": "value >= 2", "question": "Times in last year hurt someone physically"},
    {"csv_code": "Y2609100", "ref_id": "Y26091.00", "variable": "YASR-61B", "survey_year": 2010, "item_family": "physical_fight", "positive_rule": "value == 1", "question": "In last year got in physical fight at school or work"},
    {"csv_code": "Y2958300", "ref_id": "Y29583.00", "variable": "YASR-60C", "survey_year": 2012, "item_family": "hurt_physically", "positive_rule": "value >= 2", "question": "Times in last year hurt someone physically"},
    {"csv_code": "Y2959200", "ref_id": "Y29592.00", "variable": "YASR-61B", "survey_year": 2012, "item_family": "physical_fight", "positive_rule": "value == 1", "question": "In last year got in physical fight at school or work"},
    {"csv_code": "Y3325701", "ref_id": "Y33257.01", "variable": "YASR-60B-J~000002", "survey_year": 2014, "item_family": "hurt_physically", "positive_rule": "value >= 2", "question": "Times in last year hurt someone physically"},
    {"csv_code": "Y3670801", "ref_id": "Y36708.01", "variable": "YASR-60B-J~000002", "survey_year": 2016, "item_family": "hurt_physically", "positive_rule": "value >= 2", "question": "Times in last year hurt someone physically"},
    {"csv_code": "Y4275601", "ref_id": "Y42756.01", "variable": "YASR-60B-J~000002", "survey_year": 2018, "item_family": "hurt_physically", "positive_rule": "value >= 2", "question": "Times in last year hurt someone physically"},
    {"csv_code": "Y4596801", "ref_id": "Y45968.01", "variable": "YASR-60B-J~000002", "survey_year": 2020, "item_family": "hurt_physically", "positive_rule": "value >= 2", "question": "Times in last year hurt someone physically"},
]

item_meta = pd.DataFrame(ITEMS)
item_meta.to_csv(OUT_DIR / "violence_age15_25_target_item_metadata.csv", index=False)
safe_display(item_meta)


,csv_code,ref_id,variable,survey_year,item_family,positive_rule,question
0,C0730700,C07307.00,CS884221,1988,hurt_need_doctor,value >= 1,Hurt someone bad enough to need a doctor
1,C0942400,C09424.00,CS906613,1990,hurt_need_doctor,value >= 1,Number of times hurt someone enough to need a ...
2,Y0375300,Y03753.00,YASR-61B,1994,physical_fight,value == 1,In past year got in fight at school or work
3,Y0375900,Y03759.00,YASR-61I,1994,attacked_to_hurt,value == 1,In past year attacked someone to seriously hur...
4,Y0376500,Y03765.00,YASR-61O,1994,hurt_need_doctor,value == 1,In past year hurt someone enough to need doctor
5,Y0669000,Y06690.00,YASR-61B,1996,physical_fight,value == 1,In past year got in fight at school or work
6,Y0669600,Y06696.00,YASR-61I,1996,attacked_to_hurt,value == 1,In past year attacked someone to seriously hur...
7,Y0670200,Y06702.00,YASR-61O,1996,hurt_need_doctor,value == 1,In past year hurt someone enough to need doctor
8,Y0966400,Y09664.00,YASR-61B,1998,physical_fight,value == 1,In past year got in fight at school or work
9,Y0967000,Y09670.00,YASR-61I,1998,attacked_to_hurt,value == 1,In past year attacked someone to seriously hur...


## Load and Merge Demographics

The uploaded violence extract already contains basic identifiers, race, sex, and birth year. We additionally merge selected demographic/background fields from the main feature file for descriptive analyses.


In [3]:
violence = pd.read_csv(VIOLENCE_PATH)
main_demo_cols = [
    "C0000100", "C0000200", "C0005300", "C0005400", "C0005500", "C0005700", "C0005800", "C0007000"
]
main_demo_cols = [c for c in main_demo_cols if c in pd.read_csv(MAIN_FEATURES_PATH, nrows=0).columns]
main_demo = pd.read_csv(MAIN_FEATURES_PATH, usecols=main_demo_cols)

# Prefer the richer main feature file for demographics, but keep violence extract values if needed.
analysis_base = violence.merge(main_demo, on="C0000100", how="left", suffixes=("", "_main"))
for col in ["C0000200", "C0005300", "C0005400", "C0005700"]:
    main_col = f"{col}_main"
    if main_col in analysis_base.columns:
        analysis_base[col] = analysis_base[main_col].where(analysis_base[main_col].notna(), analysis_base[col])
        analysis_base = analysis_base.drop(columns=[main_col])

print("Violence rows:", len(violence))
print("Merged rows:", len(analysis_base))
print("Unique children:", analysis_base["C0000100"].nunique())
safe_display(analysis_base[[c for c in ["C0000100", "C0000200", "C0005300", "C0005400", "C0005700", "C0005800", "C0007000"] if c in analysis_base.columns]].head())


Violence rows: 11551
Merged rows: 11551
Unique children: 11551


,C0000100,C0000200,C0005300,C0005400,C0005700,C0005800,C0007000
0,201,2.0,3.0,2.0,1993.0,1.0,34.0
1,202,2.0,3.0,2.0,1994.0,2.0,35.0
2,301,3.0,3.0,2.0,1981.0,1.0,19.0
3,302,3.0,3.0,2.0,1983.0,2.0,22.0
4,303,3.0,3.0,2.0,1986.0,3.0,24.0


## Construct Person-Year Violence Events

For every violence item we compute:

- `age = survey_year - birth_year`
- `observed = non-missing item response`
- `qualifying_event = positive item response`

Then we collapse to one row per child and age-year. If any item at that age is positive, that age-year counts as a qualifying violence event.


In [4]:
def clean_numeric(s):
    x = pd.to_numeric(s, errors="coerce")
    x = x.replace([np.inf, -np.inf], np.nan)
    return x.mask(x.isin(MISSING_CODES), np.nan)


def positive_from_rule(value, rule):
    if pd.isna(value):
        return np.nan
    if rule == "value == 1":
        return int(value == 1)
    if rule == "value >= 1":
        return int(value >= 1)
    if rule == "value >= 2":
        return int(value >= 2)
    raise ValueError(f"Unknown rule: {rule}")

long_rows = []
for item in ITEMS:
    code = item["csv_code"]
    if code not in analysis_base.columns:
        continue
    value = clean_numeric(analysis_base[code])
    birth_year = clean_numeric(analysis_base["C0005700"])
    tmp = pd.DataFrame({
        "C0000100": analysis_base["C0000100"],
        "C0000200": analysis_base["C0000200"],
        "birth_year": birth_year,
        "survey_year": item["survey_year"],
        "age": item["survey_year"] - birth_year,
        "csv_code": code,
        "ref_id": item["ref_id"],
        "variable": item["variable"],
        "item_family": item["item_family"],
        "question": item["question"],
        "value": value,
    })
    tmp["observed"] = tmp["value"].notna()
    tmp["qualifying_event"] = [positive_from_rule(v, item["positive_rule"]) for v in tmp["value"]]
    long_rows.append(tmp)

item_long = pd.concat(long_rows, ignore_index=True)
item_long["in_age_window_15_25"] = item_long["age"].between(AGE_START, AGE_END, inclusive="both")
item_long["pre_predictor_cutoff"] = item_long["age"] < PREDICTOR_CUTOFF_AGE

person_age = (
    item_long[item_long["observed"]]
    .groupby(["C0000100", "C0000200", "birth_year", "survey_year", "age"], dropna=False)
    .agg(
        n_items_observed=("observed", "sum"),
        any_qualifying_event=("qualifying_event", "max"),
        positive_items=("qualifying_event", "sum"),
        item_codes=("csv_code", lambda x: ";".join(sorted(set(map(str, x))))),
        item_families=("item_family", lambda x: ";".join(sorted(set(map(str, x))))),
    )
    .reset_index()
)
person_age["in_age_window_15_25"] = person_age["age"].between(AGE_START, AGE_END, inclusive="both")
person_age["pre_predictor_cutoff"] = person_age["age"] < PREDICTOR_CUTOFF_AGE

print("Observed item rows:", int(item_long["observed"].sum()))
print("Person-age observed rows:", len(person_age))
print("Observed positive person-age rows:", int(person_age["any_qualifying_event"].sum()))
safe_display(person_age.head(12))


Observed item rows: 51904
Person-age observed rows: 43180
Observed positive person-age rows: 3740


,C0000100,C0000200,birth_year,survey_year,age,n_items_observed,any_qualifying_event,positive_items,item_codes,item_families,in_age_window_15_25,pre_predictor_cutoff
0,301,3.0,1981.0,1996,15.0,3,0.0,0.0,Y0669000;Y0669600;Y0670200,attacked_to_hurt;hurt_need_doctor;physical_fight,True,False
1,301,3.0,1981.0,1998,17.0,3,1.0,1.0,Y0966400;Y0967000;Y0967600,attacked_to_hurt;hurt_need_doctor;physical_fight,True,False
2,301,3.0,1981.0,2002,21.0,1,0.0,0.0,Y1416800,physical_fight,True,False
3,301,3.0,1981.0,2008,27.0,1,0.0,0.0,Y2257500,physical_fight,False,False
4,301,3.0,1981.0,2010,29.0,1,0.0,0.0,Y2609100,physical_fight,False,False
5,301,3.0,1981.0,2012,31.0,1,0.0,0.0,Y2959200,physical_fight,False,False
6,302,3.0,1983.0,2000,17.0,1,0.0,0.0,Y1176200,physical_fight,True,False
7,302,3.0,1983.0,2002,19.0,1,0.0,0.0,Y1416800,physical_fight,True,False
8,302,3.0,1983.0,2008,25.0,1,0.0,0.0,Y2257500,physical_fight,True,False
9,302,3.0,1983.0,2010,27.0,1,0.0,0.0,Y2609100,physical_fight,False,False


## Target Construction

A person is eligible if they have at least one observed violence item at age 25 or later. This operationalizes “observed through at least age 25” using the violence-item panel.

Among eligible people, the age-standardized target is positive if the person has at least 2 distinct observed positive age-years between ages 15 and 25.


In [5]:
# Person-level observation summary.
obs_summary = (
    person_age
    .groupby("C0000100")
    .agg(
        first_observed_age=("age", "min"),
        last_observed_age=("age", "max"),
        n_observed_age_years=("age", "nunique"),
        n_observed_age_years_15_25=("in_age_window_15_25", "sum"),
        any_observed_age_ge_25=("age", lambda x: bool((x >= AGE_END).any())),
    )
    .reset_index()
)

window_events = person_age[person_age["in_age_window_15_25"]].copy()
positive_age_summary = (
    window_events[window_events["any_qualifying_event"].eq(1)]
    .groupby("C0000100")
    .agg(
        violence_positive_age_year_count_15_25=("age", "nunique"),
        violence_first_event_age_15_25=("age", "min"),
        violence_second_event_age_15_25=("age", lambda x: sorted(set(x))[1] if len(set(x)) >= 2 else np.nan),
        violence_last_event_age_15_25=("age", "max"),
        violence_positive_survey_years_15_25=("survey_year", lambda x: ";".join(map(str, sorted(set(map(int, x))))))
    )
    .reset_index()
)

window_obs_summary = (
    window_events
    .groupby("C0000100")
    .agg(
        observed_age_year_count_15_25=("age", "nunique"),
        observed_survey_years_15_25=("survey_year", lambda x: ";".join(map(str, sorted(set(map(int, x))))))
    )
    .reset_index()
)

target_df = analysis_base[[c for c in ["C0000100", "C0000200", "C0005300", "C0005400", "C0005700", "C0005800", "C0007000"] if c in analysis_base.columns]].copy()
target_df = target_df.merge(obs_summary, on="C0000100", how="left")
target_df = target_df.merge(window_obs_summary, on="C0000100", how="left")
target_df = target_df.merge(positive_age_summary, on="C0000100", how="left")

target_df["eligible_observed_through_age_25"] = target_df["any_observed_age_ge_25"].fillna(False)
target_df["violence_positive_age_year_count_15_25"] = target_df["violence_positive_age_year_count_15_25"].fillna(0).astype(int)
target_df["persistent_violence_contact_age15_25"] = np.where(
    target_df["eligible_observed_through_age_25"],
    (target_df["violence_positive_age_year_count_15_25"] >= 2).astype(int),
    np.nan,
)
target_df["predictor_cutoff_age"] = PREDICTOR_CUTOFF_AGE
target_df["predictor_cutoff_year"] = pd.to_numeric(target_df["C0005700"], errors="coerce") + PREDICTOR_CUTOFF_AGE

target_df.to_csv(OUT_DIR / "persistent_violence_contact_age15_25_targets.csv", index=False)
person_age.to_csv(OUT_DIR / "violence_person_age_events.csv", index=False)
item_long.to_csv(OUT_DIR / "violence_item_long.csv", index=False)

eligible = target_df[target_df["eligible_observed_through_age_25"]].copy()
base_rate = eligible["persistent_violence_contact_age15_25"].mean()
summary_rows = pd.DataFrame([
    {"metric": "total_children_in_extract", "value": len(target_df)},
    {"metric": "eligible_observed_through_age_25", "value": len(eligible)},
    {"metric": "not_eligible", "value": len(target_df) - len(eligible)},
    {"metric": "positive_n", "value": int(eligible["persistent_violence_contact_age15_25"].sum())},
    {"metric": "negative_n", "value": int((eligible["persistent_violence_contact_age15_25"] == 0).sum())},
    {"metric": "base_rate", "value": base_rate},
])
summary_rows.to_csv(OUT_DIR / "persistent_violence_contact_age15_25_base_rate.csv", index=False)

safe_display(summary_rows)
safe_display(target_df.head(10))


/var/folders/r9/0jdsw2x14j92gqns9s1tdybc0000gp/T/ipykernel_76888/3647304358.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  target_df["eligible_observed_through_age_25"] = target_df["any_observed_age_ge_25"].fillna(False)


,metric,value
0,total_children_in_extract,11551.000000
1,eligible_observed_through_age_25,4482.000000
2,not_eligible,7069.000000
3,positive_n,532.000000
4,negative_n,3950.000000
5,base_rate,0.118697


,C0000100,C0000200,C0005300,C0005400,C0005700,C0005800,C0007000,first_observed_age,last_observed_age,n_observed_age_years,n_observed_age_years_15_25,any_observed_age_ge_25,observed_age_year_count_15_25,observed_survey_years_15_25,violence_positive_age_year_count_15_25,violence_first_event_age_15_25,violence_second_event_age_15_25,violence_last_event_age_15_25,violence_positive_survey_years_15_25,eligible_observed_through_age_25,persistent_violence_contact_age15_25,predictor_cutoff_age,predictor_cutoff_year
0,201,2.0,3.0,2.0,1993.0,1.0,34.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,False,NaN,15,2008.0
1,202,2.0,3.0,2.0,1994.0,2.0,35.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,False,NaN,15,2009.0
2,301,3.0,3.0,2.0,1981.0,1.0,19.0,15.0,31.0,6.0,3.0,True,3.0,1996;1998;2002,1,17.0,NaN,17.0,1998,True,0.0,15,1996.0
3,302,3.0,3.0,2.0,1983.0,2.0,22.0,17.0,27.0,4.0,3.0,True,3.0,2000;2002;2008,0,NaN,NaN,NaN,NaN,True,0.0,15,1998.0
4,303,3.0,3.0,2.0,1986.0,3.0,24.0,16.0,26.0,4.0,3.0,True,3.0,2002;2008;2010,0,NaN,NaN,NaN,NaN,True,0.0,15,2001.0
5,401,4.0,3.0,1.0,1980.0,1.0,18.0,10.0,18.0,3.0,2.0,False,2.0,1996;1998,2,16.0,18.0,18.0,1996;1998,False,NaN,15,1995.0
6,403,4.0,3.0,2.0,1997.0,2.0,34.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,False,NaN,15,2012.0
7,801,8.0,3.0,2.0,1976.0,1.0,17.0,12.0,34.0,10.0,3.0,True,3.0,1994;1996;2000,3,18.0,20.0,24.0,1994;1996;2000,True,1.0,15,1991.0
8,802,8.0,3.0,1.0,1979.0,2.0,20.0,11.0,33.0,10.0,6.0,True,6.0,1994;1996;1998;2000;2002;2004,4,15.0,17.0,21.0,1994;1996;1998;2000,True,1.0,15,1994.0
9,803,8.0,3.0,2.0,1982.0,3.0,24.0,16.0,30.0,8.0,5.0,True,5.0,1998;2000;2002;2004;2006,1,18.0,NaN,18.0,2000,True,0.0,15,1997.0


## Descriptive Analyses of the Eligible Target Sample

The tables below describe the analytic sample and compare positives vs. negatives. Demographic labels are added for readability.


In [6]:
RACE_LABELS = {1: "Hispanic", 2: "Black", 3: "Non-Black/non-Hispanic"}
SEX_LABELS = {1: "Male", 2: "Female"}

desc = eligible.copy()
desc["race_label"] = pd.to_numeric(desc["C0005300"], errors="coerce").map(RACE_LABELS).fillna("Missing/invalid")
desc["sex_label"] = pd.to_numeric(desc["C0005400"], errors="coerce").map(SEX_LABELS).fillna("Missing/invalid")
desc["birth_cohort_5yr"] = (pd.to_numeric(desc["C0005700"], errors="coerce") // 5 * 5).astype("Int64").astype(str) + "-" + ((pd.to_numeric(desc["C0005700"], errors="coerce") // 5 * 5 + 4).astype("Int64").astype(str))

def subgroup_table(df, group_col):
    out = (
        df.groupby(group_col, dropna=False)
        .agg(
            n=("C0000100", "size"),
            positive_n=("persistent_violence_contact_age15_25", "sum"),
            base_rate=("persistent_violence_contact_age15_25", "mean"),
            mean_positive_age_years=("violence_positive_age_year_count_15_25", "mean"),
            mean_observed_age_years_15_25=("observed_age_year_count_15_25", "mean"),
        )
        .reset_index()
    )
    out["negative_n"] = out["n"] - out["positive_n"]
    return out[[group_col, "n", "positive_n", "negative_n", "base_rate", "mean_positive_age_years", "mean_observed_age_years_15_25"]]

by_sex = subgroup_table(desc, "sex_label")
by_race = subgroup_table(desc, "race_label")
by_birth_cohort = subgroup_table(desc, "birth_cohort_5yr")

age_event_distribution = (
    desc.groupby("violence_positive_age_year_count_15_25")
    .size()
    .reset_index(name="n")
)
age_event_distribution["percent"] = age_event_distribution["n"] / len(desc)

first_second_age_summary = desc[["violence_first_event_age_15_25", "violence_second_event_age_15_25", "violence_last_event_age_15_25", "last_observed_age", "observed_age_year_count_15_25"]].describe().T

by_sex.to_csv(OUT_DIR / "target_descriptives_by_sex.csv", index=False)
by_race.to_csv(OUT_DIR / "target_descriptives_by_race.csv", index=False)
by_birth_cohort.to_csv(OUT_DIR / "target_descriptives_by_birth_cohort.csv", index=False)
age_event_distribution.to_csv(OUT_DIR / "target_positive_age_year_distribution.csv", index=False)
first_second_age_summary.to_csv(OUT_DIR / "target_event_age_descriptive_summary.csv")

print("By sex")
safe_display(by_sex)
print("By race")
safe_display(by_race)
print("By birth cohort")
safe_display(by_birth_cohort)
print("Positive age-year count distribution")
safe_display(age_event_distribution)
print("Event/observation age summary")
safe_display(first_second_age_summary)


By sex


,sex_label,n,positive_n,negative_n,base_rate,mean_positive_age_years,mean_observed_age_years_15_25
0,Female,2284,175.0,2109.0,0.07662,0.370403,4.673749
1,Male,2198,357.0,1841.0,0.16242,0.641947,4.497217


By race


,race_label,n,positive_n,negative_n,base_rate,mean_positive_age_years,mean_observed_age_years_15_25
0,Black,1686,236.0,1450.0,0.139976,0.570581,4.480990
1,Hispanic,1034,108.0,926.0,0.104449,0.475822,4.524510
2,Non-Black/non-Hispanic,1762,188.0,1574.0,0.106697,0.455732,4.726122


By birth cohort


,birth_cohort_5yr,n,positive_n,negative_n,base_rate,mean_positive_age_years,mean_observed_age_years_15_25
0,1970-1974,83,10.0,73.0,0.120482,0.626506,2.616438
1,1975-1979,1020,187.0,833.0,0.183333,0.692157,3.991960
2,1980-1984,2149,244.0,1905.0,0.113541,0.489530,4.569811
3,1985-1989,1230,91.0,1139.0,0.073984,0.363415,5.218419


Positive age-year count distribution


,violence_positive_age_year_count_15_25,n,percent
0,0,2891,0.645025
1,1,1059,0.236278
2,2,422,0.094154
3,3,88,0.019634
4,4,20,0.004462
5,5,2,0.000446


Event/observation age summary


,count,mean,std,min,25%,50%,75%,max
violence_first_event_age_15_25,1591.0,16.887492,2.120707,15.0,15.0,16.0,18.0,25.0
violence_second_event_age_15_25,532.0,18.990602,2.164146,17.0,17.0,18.0,20.0,25.0
violence_last_event_age_15_25,1591.0,18.081710,2.594307,15.0,16.0,18.0,19.0,25.0
last_observed_age,4482.0,29.186524,3.042072,25.0,27.0,29.0,30.0,42.0
observed_age_year_count_15_25,4415.0,4.587542,1.314407,1.0,4.0,5.0,6.0,6.0


## Item-Level Missingness and Positivity

This table checks which violence items contribute most to the target and how much missingness they have in the full extract and among the eligible age-window sample.


In [7]:
item_rows = []
for item in ITEMS:
    code = item["csv_code"]
    tmp = item_long[item_long["csv_code"].eq(code)].copy()
    eligible_window_ids = set(desc["C0000100"])
    tmp_eligible_window = tmp[tmp["C0000100"].isin(eligible_window_ids) & tmp["in_age_window_15_25"]]
    item_rows.append({
        "csv_code": code,
        "ref_id": item["ref_id"],
        "variable": item["variable"],
        "survey_year": item["survey_year"],
        "age_range_in_extract_min": tmp["age"].min(),
        "age_range_in_extract_max": tmp["age"].max(),
        "item_family": item["item_family"],
        "question": item["question"],
        "full_n": len(tmp),
        "full_observed_n": int(tmp["observed"].sum()),
        "full_missing_rate": 1 - float(tmp["observed"].mean()),
        "full_positive_n": int(tmp["qualifying_event"].fillna(0).sum()),
        "eligible_window_n": len(tmp_eligible_window),
        "eligible_window_observed_n": int(tmp_eligible_window["observed"].sum()),
        "eligible_window_missing_rate": 1 - float(tmp_eligible_window["observed"].mean()) if len(tmp_eligible_window) else np.nan,
        "eligible_window_positive_n": int(tmp_eligible_window["qualifying_event"].fillna(0).sum()),
        "eligible_window_positive_rate_among_observed": float(tmp_eligible_window.loc[tmp_eligible_window["observed"], "qualifying_event"].mean()) if int(tmp_eligible_window["observed"].sum()) else np.nan,
    })

item_missingness = pd.DataFrame(item_rows)
item_missingness.to_csv(OUT_DIR / "violence_item_missingness_and_positivity.csv", index=False)
safe_display(item_missingness)


,csv_code,ref_id,variable,survey_year,age_range_in_extract_min,age_range_in_extract_max,item_family,question,full_n,full_observed_n,full_missing_rate,full_positive_n,eligible_window_n,eligible_window_observed_n,eligible_window_missing_rate,eligible_window_positive_n,eligible_window_positive_rate_among_observed
0,C0730700,C07307.00,CS884221,1988,-26.0,18.0,hurt_need_doctor,Hurt someone bad enough to need a doctor,11551,800,0.930742,152,35,30,0.142857,6,0.200000
1,C0942400,C09424.00,CS906613,1990,-24.0,20.0,hurt_need_doctor,Number of times hurt someone enough to need a ...,11551,1136,0.901654,235,194,135,0.304124,31,0.229630
2,Y0375300,Y03753.00,YASR-61B,1994,-20.0,24.0,physical_fight,In past year got in fight at school or work,11551,912,0.921046,288,1103,865,0.215775,278,0.321387
3,Y0375900,Y03759.00,YASR-61I,1994,-20.0,24.0,attacked_to_hurt,In past year attacked someone to seriously hur...,11551,913,0.920959,87,1103,866,0.214869,83,0.095843
4,Y0376500,Y03765.00,YASR-61O,1994,-20.0,24.0,hurt_need_doctor,In past year hurt someone enough to need doctor,11551,913,0.920959,109,1103,866,0.214869,105,0.121247
5,Y0669000,Y06690.00,YASR-61B,1996,-18.0,26.0,physical_fight,In past year got in fight at school or work,11551,1512,0.869102,420,1927,1446,0.249611,403,0.278700
6,Y0669600,Y06696.00,YASR-61I,1996,-18.0,26.0,attacked_to_hurt,In past year attacked someone to seriously hur...,11551,1514,0.868929,144,1927,1448,0.248573,140,0.096685
7,Y0670200,Y06702.00,YASR-61O,1996,-18.0,26.0,hurt_need_doctor,In past year hurt someone enough to need doctor,11551,1510,0.869275,164,1927,1444,0.250649,159,0.110111
8,Y0966400,Y09664.00,YASR-61B,1998,-16.0,28.0,physical_fight,In past year got in fight at school or work,11551,1939,0.832136,451,2814,1841,0.345771,424,0.230310
9,Y0967000,Y09670.00,YASR-61I,1998,-16.0,28.0,attacked_to_hurt,In past year attacked someone to seriously hur...,11551,1944,0.831703,141,2814,1845,0.344350,135,0.073171


## Predictor Cutoff Feasibility Check

The final target uses age 15 as a fixed baseline. For later modeling, predictors should be restricted to information before age 15. This quick check counts how many original BoL/model features are measured before each person's 15th birthday.


In [8]:
feature_index_path = PROJECT_DIR / "BoL approach" / "metadata_examples" / "child_crime_broad_persistent_feature_index.csv"
if feature_index_path.exists():
    feature_index = pd.read_csv(feature_index_path)
    feature_index["year_num"] = pd.to_numeric(feature_index["survey_year"], errors="coerce")
    feature_cols = [c for c in feature_index["csv_code"].tolist() if c in pd.read_csv(MAIN_FEATURES_PATH, nrows=0).columns]
    feature_year = feature_index.set_index("csv_code")["year_num"].to_dict()

    cutoff_check = desc[["C0000100", "C0005700", "predictor_cutoff_year", "persistent_violence_contact_age15_25"]].copy()
    counts = []
    for _, row in cutoff_check.iterrows():
        cutoff_year = row["predictor_cutoff_year"]
        n_pre15 = 0
        for col in feature_cols:
            y = feature_year.get(col)
            if pd.isna(y):
                # XRND/static variables can be used if not outcome-derived.
                n_pre15 += 1
            elif y < cutoff_year:
                n_pre15 += 1
        counts.append(n_pre15)
    cutoff_check["n_original_bol_features_pre15_by_year"] = counts
    cutoff_summary = cutoff_check.groupby("persistent_violence_contact_age15_25")["n_original_bol_features_pre15_by_year"].describe()
    cutoff_check.to_csv(OUT_DIR / "predictor_cutoff_age15_feature_availability.csv", index=False)
    cutoff_summary.to_csv(OUT_DIR / "predictor_cutoff_age15_feature_availability_summary.csv")
    safe_display(cutoff_summary)
else:
    print("Feature index not found; skipped predictor cutoff feasibility check.")


,count,mean,std,min,25%,50%,75%,max
persistent_violence_contact_age15_25,,,,,,,,
0.0,3950.0,191.546582,71.767952,6.0,150.0,201.0,250.0,283.0
1.0,532.0,161.813910,68.111277,6.0,105.0,155.0,201.0,283.0


## Final Summary

The final cell prints the key quantities needed for reporting.


In [9]:
final_summary = {
    "target_name": "persistent_violence_contact_age15_25",
    "definition": "eligible if observed through age 25; positive if >=2 distinct positive violence age-years from ages 15-25",
    "predictor_cutoff": "age 15; use predictors before 15th birthday only",
    "total_children_in_extract": int(len(target_df)),
    "eligible_n": int(len(eligible)),
    "missing_not_eligible_n": int(len(target_df) - len(eligible)),
    "positive_n": int(eligible["persistent_violence_contact_age15_25"].sum()),
    "negative_n": int((eligible["persistent_violence_contact_age15_25"] == 0).sum()),
    "base_rate": float(eligible["persistent_violence_contact_age15_25"].mean()),
    "mean_observed_age_years_15_25": float(eligible["observed_age_year_count_15_25"].mean()),
    "median_observed_age_years_15_25": float(eligible["observed_age_year_count_15_25"].median()),
}
(OUT_DIR / "persistent_violence_contact_age15_25_summary.json").write_text(json.dumps(final_summary, indent=2))
print(json.dumps(final_summary, indent=2))


{
  "target_name": "persistent_violence_contact_age15_25",
  "definition": "eligible if observed through age 25; positive if >=2 distinct positive violence age-years from ages 15-25",
  "predictor_cutoff": "age 15; use predictors before 15th birthday only",
  "total_children_in_extract": 11551,
  "eligible_n": 4482,
  "missing_not_eligible_n": 7069,
  "positive_n": 532,
  "negative_n": 3950,
  "base_rate": 0.11869701026327532,
  "mean_observed_age_years_15_25": 4.587542468856173,
  "median_observed_age_years_15_25": 5.0
}
